# First find the huc8 value/s for your area of interest/ Not necessary if you already have one from previous code

In [ ]:
import geopandas as gpd
import pandas as pd

# Load AOI shapefile
aoi = gpd.read_file("/content/boundary.shp")

# Load full HUC8 shapefile (in EPSG:5070)
huc8 = gpd.read_file("/content/HUC_8_EPSG_5070.shp")

# Ensure CRS match
if aoi.crs != huc8.crs:
    aoi = aoi.to_crs(huc8.crs)

# Spatial join: HUC8s intersecting AOI
intersecting = gpd.sjoin(huc8, aoi, how="inner", predicate="intersects")

# Extract unique HUC8 codes as sorted strings
unique_huc8_codes = sorted(intersecting["HUC8"].astype(str).unique())

# Save to CSV (with header, preserve leading zeros)
pd.DataFrame(unique_huc8_codes, columns=["HUC8"]).to_csv("HUC8.csv", index=False)

print("✅ Saved to HUC8.csv")


✅ Saved to HUC8.csv


# Find the NWM reach_IDs within the domain

In [ ]:
!pip install s3fs zarr fsspec xarray geopandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is

In [ ]:
import s3fs
import xarray as xr
import geopandas as gpd
import zarr

# === Input shapefile ===
input_model_boundary_filename = "boundary.shp"  # replace with your shapefile name

# --- Connect to NOAA NWM dataset on AWS ---
s3 = s3fs.S3FileSystem(anon=True, client_kwargs=dict(region_name="us-east-1"))
store = s3fs.S3Map(
    root="s3://noaa-nwm-retrospective-3-0-pds/CONUS/zarr/chrtout.zarr",
    s3=s3,
    check=False
)

# --- Open dataset using legacy API ---
nwm_ds = xr.open_zarr(store)   # no "engine" here
df = nwm_ds[["feature_id", "latitude", "longitude"]].to_dataframe().reset_index()

# --- Convert to GeoDataFrame ---
nwm_gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude))
nwm_gdf.crs = "EPSG:4326"

# --- Read your model boundary shapefile ---
boundary = gpd.read_file(input_model_boundary_filename)

# --- Ensure both layers have the same CRS ---
if boundary.crs != "EPSG:4326":
    boundary = boundary.to_crs("EPSG:4326")

# --- Spatial join: keep only NWM reaches inside boundary ---
reach_ids_gdf = gpd.sjoin(nwm_gdf, boundary, how="inner", predicate="intersects")

# --- Extract only required columns ---
all_reaches = reach_ids_gdf[["feature_id", "latitude", "longitude"]].drop_duplicates()

# --- Save to CSV ---
all_reaches.to_csv("feature_IDs.csv", index=False)

print("✅ File 'feature_IDs.csv' created with all Feature IDs and coordinates in your shapefile domain.")


✅ File 'feature_IDs.csv' created with all Feature IDs and coordinates in your shapefile domain.


In [ ]:
!find /content -type f ! -name '*.csv' -delete


### Make an input files directory

In [ ]:
!mkdir /content/input_raster_files


### Upload all the input raster files to content section and run the following code to move them to input folder.

In [ ]:
import os
import shutil
import glob

# Define target folder
target_folder = '/content/input_raster_files'
os.makedirs(target_folder, exist_ok=True)

# Get all non-CSV files in /content
files_to_move = [f for f in glob.glob('/content/*') if os.path.isfile(f) and not f.endswith('.csv')]

# Move each file
for file_path in files_to_move:
    filename = os.path.basename(file_path)
    if not file_path.startswith(target_folder):  # Avoid moving into itself
        shutil.move(file_path, os.path.join(target_folder, filename))

print("✅ Files moved to:", target_folder)



✅ Files moved to: /content/input_raster_files


## Run this code to deal with the no data regions

In [ ]:
!pip install geopandas rasterio fiona shapely pyproj

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 86.1 MB/s eta 0:00:00


## This code is for dealing with resolutions (keeps minimum) of input raster maps and also convert to float 32 type.

In [ ]:
import rasterio
from rasterio.enums import Resampling
import os
import glob
import numpy as np

# Folders
input_folder = "input_raster_files/"
output_folder = "resampled_output/"
os.makedirs(output_folder, exist_ok=True)

# --- 1) Discover inputs and the finest (minimum) pixel size ---
raster_files = glob.glob(os.path.join(input_folder, "*.tif"))
print(f"Found {len(raster_files)} rasters to process.")

if not raster_files:
    raise SystemExit("No rasters found.")

# Ensure all rasters share the same CRS (recommended)
first_crs = None
finest = None
for rp in raster_files:
    with rasterio.open(rp) as s:
        if first_crs is None:
            first_crs = s.crs
        elif s.crs != first_crs:
            raise ValueError(f"CRS mismatch between rasters. '{os.path.basename(rp)}' differs.")

        px_x = abs(s.transform.a)
        px_y = abs(s.transform.e)

        # target is isotropic: choose the smallest pixel dimension seen anywhere
        min_here = min(px_x, px_y)
        finest = min_here if finest is None else min(finest, min_here)

print(f"Target (finest) resolution: {finest}")

# --- 2) Process each raster to the finest resolution ---
for raster_path in raster_files:
    filename = os.path.basename(raster_path)
    output_path = os.path.join(output_folder, filename)  # Keep same name

    with rasterio.open(raster_path) as src:
        transform = src.transform
        current_res_x = abs(transform.a)
        current_res_y = abs(transform.e)

        # Scale factors to reach the finest resolution (may upsample)
        scale_x = current_res_x / finest
        scale_y = current_res_y / finest

        new_width = max(1, int(round(src.width * scale_x)))
        new_height = max(1, int(round(src.height * scale_y)))

        # New transform (use ratios of old/new sizes)
        new_transform = transform * transform.scale(
            src.width / new_width,
            src.height / new_height
        )

        # Resample and convert to float32
        data = src.read(
            out_shape=(src.count, new_height, new_width),
            resampling=Resampling.bilinear
        ).astype(np.float32)

        # Update metadata
        profile = src.meta.copy()
        profile.update({
            'height': new_height,
            'width': new_width,
            'transform': new_transform,
            'dtype': 'float32',
            'nodata': float(src.nodata) if src.nodata is not None else -9999.0
        })

    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(data)

    print(f"✅ Finished: {filename}")

print("🎉 Done: All rasters processed and saved with original filenames.")


Found 12 rasters to process.
Target (finest) resolution: 9.969067400416828
✅ Finished: AutoRoute_03140201_240_cms_depth.tif
✅ Finished: HAND_03140201_240_cms_depth.tif
✅ Finished: AutoRoute_03140201_209_cms_depth.tif
✅ Finished: AutoRoute_03140201_130_cms_depth.tif
✅ Finished: HAND_03140201_209_cms_depth.tif
✅ Finished: HAND_03140201_272_cms_depth.tif
✅ Finished: HAND_03140201_165_cms_depth.tif
✅ Finished: AutoRoute_03140201_272_cms_depth.tif
✅ Finished: HAND_03140201_130_cms_depth.tif
✅ Finished: AutoRoute_03140201_165_cms_depth.tif
✅ Finished: HAND_03140201_79_cms_depth.tif
✅ Finished: AutoRoute_03140201_79_cms_depth.tif
🎉 Done: All rasters processed and saved with original filenames.


In [ ]:
# @title Run this code to convert all the nodata region values to -99999
import os
import rasterio
import numpy as np
import csv

# Directories
input_dir = '/content/resampled_output'
output_dir = '/content/corrected_rasters'
os.makedirs(output_dir, exist_ok=True)
log_path = os.path.join(output_dir, 'nodata_log.csv')

# Fixed NoData value expected by the visualization tool
FIXED_NODATA = -99999

# Common bad values to treat as NoData
COMMON_BAD_VALUES = [-9999, 0, -3.402823e+38]

# Initialize log file
with open(log_path, 'w', newline='') as log_file:
    writer = csv.writer(log_file)
    writer.writerow(['File Name', 'Original NoData', 'Unusual Values < -1e6'])

# Process all .tif files in input directory
for file_name in os.listdir(input_dir):
    if file_name.endswith(".tif"):
        input_path = os.path.join(input_dir, file_name)
        output_path = os.path.join(output_dir, file_name)

        with rasterio.open(input_path) as src:
            data = src.read(1).astype(np.float32)
            profile = src.profile
            original_nodata = src.nodata

            # Step 1: Convert existing -99999 to NaN (cleanup)
            data[data == FIXED_NODATA] = np.nan

            # Step 2: Build mask for invalid values
            mask = np.isin(data, COMMON_BAD_VALUES)
            if original_nodata is not None:
                if np.isnan(original_nodata):
                    mask |= np.isnan(data)
                else:
                    mask |= (data == original_nodata)
            mask |= np.isnan(data)

            # Step 3: Replace all masked values with -99999
            data[mask] = FIXED_NODATA

            # Step 4: Update profile and save without reprojection
            profile.update(nodata=FIXED_NODATA, compress='LZW')

            with rasterio.open(output_path, 'w', **profile) as dst:
                dst.write(data, 1)

            # Step 5: Log NoData info and unusual values
            unusual_values = np.unique(data[data < -1e6])
            with open(log_path, 'a', newline='') as log_file:
                writer = csv.writer(log_file)
                writer.writerow([file_name, original_nodata, "; ".join(map(str, unusual_values))])

print("✅ All rasters cleaned and saved with NoData = -99999.")
print(f"📄 Log saved to: {log_path}")


✅ All rasters cleaned and saved with NoData = -99999.
📄 Log saved to: /content/corrected_rasters/nodata_log.csv


## Task 0: Covert depth rasters to extent vectors.

In [ ]:
import os
import rasterio
from rasterio.features import shapes
import geopandas as gpd
import numpy as np

# Input and output directories
input_folder = '/content/corrected_rasters'  # ✅ Update as needed
output_folder = '/content/extent_files'
os.makedirs(output_folder, exist_ok=True)

# Loop through all raster files
for filename in os.listdir(input_folder):
    if filename.endswith('depth.tif'):  # Adjust pattern as needed
        raster_path = os.path.join(input_folder, filename)

        with rasterio.open(raster_path) as src:
            image = src.read(1)
            raster_crs = src.crs

            # Step 1: Detect or define NoData value
            nodata_val = src.nodata
            if nodata_val is None:
                nodata_val = -99999  # fallback (common placeholder)

            # Step 2: Build mask to ignore NoData, zeros, and NaNs
            mask = (~np.isnan(image)) & (image != 0) & (image != nodata_val)

            # Step 3: Extract shapes while safely converting values
            results = (
                {
                    "properties": {"value": float(v) if not np.isnan(v) else None},
                    "geometry": s
                }
                for s, v in shapes(image, mask=mask, transform=src.transform)
                if not np.isnan(v)
            )

            geoms = list(results)
            if not geoms:
                print(f"⚠️ Skipping {filename}: No valid geometries found.")
                continue

            # Step 4: Create GeoDataFrame
            gdf = gpd.GeoDataFrame.from_features(geoms)
            gdf.set_crs(raster_crs, inplace=True)

            # Step 5: Generate output name and save shapefile
            base_name = os.path.splitext(filename)[0].replace('_depth', '')
            out_name = f"{base_name}_extent.shp"
            out_path = os.path.join(output_folder, out_name)

            gdf.to_file(out_path)
            print(f"✅ Saved vector: {out_name}")

print(f"\n🎉 Conversion complete. Vector files saved to: {output_folder}")


✅ Saved vector: AutoRoute_03140201_240_cms_extent.shp
✅ Saved vector: HAND_03140201_240_cms_extent.shp
✅ Saved vector: AutoRoute_03140201_209_cms_extent.shp
✅ Saved vector: AutoRoute_03140201_130_cms_extent.shp
✅ Saved vector: HAND_03140201_209_cms_extent.shp
✅ Saved vector: HAND_03140201_272_cms_extent.shp
✅ Saved vector: HAND_03140201_165_cms_extent.shp
✅ Saved vector: AutoRoute_03140201_272_cms_extent.shp
✅ Saved vector: HAND_03140201_130_cms_extent.shp
✅ Saved vector: AutoRoute_03140201_165_cms_extent.shp
✅ Saved vector: HAND_03140201_79_cms_extent.shp
✅ Saved vector: AutoRoute_03140201_79_cms_extent.shp

🎉 Conversion complete. Vector files saved to: /content/extent_files


# Task 1: Project all the GIS files to the EPSG:4326 projection system.

In [ ]:
import os
import glob
import shutil
import geopandas as gpd
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling

# Input directories
vector_input_dir = '/content/extent_files'
raster_input_dir = '/content/corrected_rasters'
output_dir = '/content/Projected_files'

# Remove existing output directory and create a new one
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir)

# Target CRS
target_crs = 'EPSG:4326'

# ---------------------------
# Reproject VECTOR files
# ---------------------------
vector_files = glob.glob(os.path.join(vector_input_dir, '*.shp'))

for shp in vector_files:
    try:
        gdf = gpd.read_file(shp)
        print(f"\nVector file: {os.path.basename(shp)}")
        print(f"Original CRS: {gdf.crs}")

        if gdf.crs is None:
            print("❌ CRS not defined. Skipping.")
            continue

        if gdf.crs.to_string() != target_crs:
            gdf_proj = gdf.to_crs(target_crs)
        else:
            gdf_proj = gdf

        # Save projected shapefile directly into the output folder
        basename = os.path.splitext(os.path.basename(shp))[0]
        out_path = os.path.join(output_dir, f"{basename}.shp")
        gdf_proj.to_file(out_path)
        print("✅ Reprojected and saved.")
    except Exception as e:
        print(f"❌ Error with {shp}: {e}")

# ---------------------------
# Reproject RASTER files
# ---------------------------
raster_files = glob.glob(os.path.join(raster_input_dir, '*.tif'))

for raster in raster_files:
    try:
        with rasterio.open(raster) as src:
            print(f"\nRaster file: {os.path.basename(raster)}")
            print(f"Original CRS: {src.crs}")

            if src.crs is None:
                print("❌ CRS not defined. Skipping.")
                continue

            output_path = os.path.join(output_dir, os.path.basename(raster))

            if src.crs.to_string() != target_crs:
                transform, width, height = calculate_default_transform(
                    src.crs, target_crs, src.width, src.height, *src.bounds
                )
                kwargs = src.meta.copy()
                kwargs.update({
                    'crs': target_crs,
                    'transform': transform,
                    'width': width,
                    'height': height
                })

                with rasterio.open(output_path, 'w', **kwargs) as dst:
                    for i in range(1, src.count + 1):
                        reproject(
                            source=rasterio.band(src, i),
                            destination=rasterio.band(dst, i),
                            src_transform=src.transform,
                            src_crs=src.crs,
                            dst_transform=transform,
                            dst_crs=target_crs,
                            resampling=Resampling.nearest
                        )
                print("✅ Reprojected and saved.")
            else:
                shutil.copy(raster, output_path)
                print("ℹ️ Already in target CRS. Copied.")
    except Exception as e:
        print(f"❌ Error with {raster}: {e}")





Vector file: HAND_03140201_240_cms_extent.shp
Original CRS: EPSG:5070
✅ Reprojected and saved.

Vector file: AutoRoute_03140201_79_cms_extent.shp
Original CRS: EPSG:5070
✅ Reprojected and saved.

Vector file: HAND_03140201_79_cms_extent.shp
Original CRS: EPSG:5070
✅ Reprojected and saved.

Vector file: HAND_03140201_272_cms_extent.shp
Original CRS: EPSG:5070
✅ Reprojected and saved.

Vector file: HAND_03140201_165_cms_extent.shp
Original CRS: EPSG:5070
✅ Reprojected and saved.

Vector file: HAND_03140201_130_cms_extent.shp
Original CRS: EPSG:5070
✅ Reprojected and saved.

Vector file: HAND_03140201_209_cms_extent.shp
Original CRS: EPSG:5070
✅ Reprojected and saved.

Vector file: AutoRoute_03140201_209_cms_extent.shp
Original CRS: EPSG:5070
✅ Reprojected and saved.

Vector file: AutoRoute_03140201_272_cms_extent.shp
Original CRS: EPSG:5070
✅ Reprojected and saved.

Vector file: AutoRoute_03140201_165_cms_extent.shp
Original CRS: EPSG:5070
✅ Reprojected and saved.

Vector file: AutoRout

# Task 2: Preprocess the projected vector files

In [ ]:
# @title Dissolve polygons, smooth boundaries, and remove all holes
import os, glob, math, shutil
import numpy as np
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union

# ---------- Directories ----------
input_dir = '/content/Projected_files'
final_dir = '/content/final_gis_files'
os.makedirs(final_dir, exist_ok=True)

# ---------- Knobs ----------
dissolve_by   = None    # e.g., "HUC8"; None = dissolve all features per file
tol_factor    = 1.5
morph_shave   = 0.0
validity_fix  = True

# ---------- Helpers ----------
def pick_metric_crs(gdf: gpd.GeoDataFrame) -> str:
    try:
        lonlat = gdf if (gdf.crs and gdf.crs.is_geographic) else gdf.to_crs(4326)
        c = lonlat.unary_union.centroid
        cx, cy = float(c.x), float(c.y)
        if -170 <= cx <= -50 and 5 <= cy <= 85:
            return "EPSG:5070"
        zone = int(math.floor((cx + 180) / 6) + 1)
        return f"EPSG:{32600 + zone}" if cy >= 0 else f"EPSG:{32700 + zone}"
    except Exception:
        return "EPSG:3857"

def median_edge_len(geom):
    lens = []
    def ring_lengths(poly: Polygon):
        xy = np.asarray(poly.exterior.coords)
        seg = np.sqrt(((xy[1:] - xy[:-1])**2).sum(axis=1))
        return seg.tolist()
    if isinstance(geom, Polygon):
        lens += ring_lengths(geom)
    elif isinstance(geom, MultiPolygon):
        for p in geom.geoms:
            lens += ring_lengths(p)
    return float(np.median(lens)) if lens else 0.0

def remove_all_holes(g):
    """Return geometry with all interior holes removed"""
    if isinstance(g, Polygon):
        return Polygon(g.exterior)
    elif isinstance(g, MultiPolygon):
        cleaned = [Polygon(p.exterior) for p in g.geoms if not p.is_empty]
        return MultiPolygon(cleaned) if len(cleaned) > 1 else (cleaned[0] if cleaned else g)
    return g

# ---------- Process all shapefiles ----------
shapefiles = glob.glob(os.path.join(input_dir, '*.shp'))
if not shapefiles:
    print(f"No .shp files found in {input_dir}")

for shp_path in shapefiles:
    try:
        base_name = os.path.splitext(os.path.basename(shp_path))[0]
        print(f"\nProcessing: {base_name}")

        gdf = gpd.read_file(shp_path)
        orig_crs = gdf.crs

        if gdf.crs is None:
            gdf.set_crs("EPSG:4326", inplace=True)

        metric_crs = pick_metric_crs(gdf)
        gdf_m = gdf.to_crs(metric_crs)

        # Dissolve
        if dissolve_by and dissolve_by in gdf_m.columns:
            dissolved = gdf_m.dissolve(by=dissolve_by, as_index=False)
        else:
            dissolved = gpd.GeoDataFrame(geometry=[unary_union(gdf_m.geometry)], crs=gdf_m.crs)

        # Simplify
        out_geoms = []
        for geom in dissolved.geometry:
            g = geom
            if morph_shave and morph_shave > 0:
                g = g.buffer(morph_shave).buffer(-morph_shave)

            tol = max(median_edge_len(g) * tol_factor, 0.0)
            g_simple = g.simplify(tol, preserve_topology=True)

            if validity_fix:
                g_simple = g_simple.buffer(0)

            # 🔹 Remove all holes
            g_simple = remove_all_holes(g_simple)

            if not g_simple.is_empty:
                out_geoms.append(g_simple)

        if not out_geoms:
            print("⚠️  No valid geometry after simplification; skipping.")
            continue

        final_metric = unary_union(out_geoms)
        out_m = gpd.GeoDataFrame(geometry=[final_metric], crs=metric_crs)

        target_crs = orig_crs if orig_crs is not None else metric_crs
        out_final = out_m.to_crs(target_crs)

        # ---------- Save shapefile ----------
        out_path = os.path.join(final_dir, f"{base_name}.shp")
        out_final.to_file(out_path)
        print(f"💾 Saved shapefile: {out_path}")

    except Exception as e:
        print(f"❌ Error processing {shp_path}: {e}")



Processing: HAND_03140201_240_cms_extent


/tmp/ipython-input-552181918.py:23: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  c = lonlat.unary_union.centroid


💾 Saved shapefile: /content/final_gis_files/HAND_03140201_240_cms_extent.shp

Processing: AutoRoute_03140201_79_cms_extent
💾 Saved shapefile: /content/final_gis_files/AutoRoute_03140201_79_cms_extent.shp

Processing: HAND_03140201_79_cms_extent


/tmp/ipython-input-552181918.py:23: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  c = lonlat.unary_union.centroid
/tmp/ipython-input-552181918.py:23: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  c = lonlat.unary_union.centroid


💾 Saved shapefile: /content/final_gis_files/HAND_03140201_79_cms_extent.shp

Processing: HAND_03140201_272_cms_extent


...

/tmp/ipython-input-552181918.py:23: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  c = lonlat.unary_union.centroid


💾 Saved shapefile: /content/final_gis_files/HAND_03140201_130_cms_extent.shp

Processing: HAND_03140201_209_cms_extent


/tmp/ipython-input-552181918.py:23: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  c = lonlat.unary_union.centroid


💾 Saved shapefile: /content/final_gis_files/HAND_03140201_209_cms_extent.shp

Processing: AutoRoute_03140201_209_cms_extent
💾 Saved shapefile: /content/final_gis_files/AutoRoute_03140201_209_cms_extent.shp

Processing: AutoRoute_03140201_272_cms_extent
💾 Saved shapefile: /content/final_gis_files/AutoRoute_03140201_272_cms_extent.shp

Processing: AutoRoute_03140201_165_cms_extent
💾 Saved shapefile: /content/final_gis_files/AutoRoute_03140201_165_cms_extent.shp

Processing: AutoRoute_03140201_130_cms_extent
💾 Saved shapefile: /content/final_gis_files/AutoRoute_03140201_130_cms_extent.shp

Processing: AutoRoute_03140201_240_cms_extent
💾 Saved shapefile: /content/final_gis_files/AutoRoute_03140201_240_cms_extent.shp


/tmp/ipython-input-552181918.py:23: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  c = lonlat.unary_union.centroid
/tmp/ipython-input-552181918.py:23: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  c = lonlat.unary_union.centroid
/tmp/ipython-input-552181918.py:23: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  c = lonlat.unary_union.centroid
/tmp/ipython-input-552181918.py:23: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  c = lonlat.unary_union.centroid
/tmp/ipython-input-552181918.py:23: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  c = lonlat.unary_union.centroid


# Task 3: Preprocess the projected raster files

In [ ]:
#@title Run this code to align the rasters to same extent and compress rasters thereafter.
import os
import glob
import rasterio
from rasterio import warp
from rasterio.enums import Resampling

# Directories
input_dir = '/content/Projected_files'
output_dir = '/content/final_gis_files'
os.makedirs(output_dir, exist_ok=True)

# Gather all rasters directly inside input_dir (no subfolders)
raster_files = glob.glob(os.path.join(input_dir, '*.tif'))

# Use first raster as reference for alignment
with rasterio.open(raster_files[0]) as ref:
    ref_crs = ref.crs
    ref_transform = ref.transform
    ref_width = ref.width
    ref_height = ref.height
    ref_profile = ref.profile.copy()
    ref_profile.update({
        'compress': 'LZW'
    })

# Process each raster
for src_path in raster_files:
    try:
        with rasterio.open(src_path) as src:
            print(f"\nProcessing: {os.path.basename(src_path)}")

            # Ensure same CRS and align to reference grid
            if src.crs != ref_crs:
                print("❌ CRS mismatch. Skipping.")
                continue

            output_path = os.path.join(output_dir, os.path.basename(src_path))
            out_meta = ref_profile.copy()
            out_meta.update({
                'dtype': src.dtypes[0],
                'count': src.count
            })

            with rasterio.open(output_path, 'w', **out_meta) as dst:
                for i in range(1, src.count + 1):
                    warp.reproject(
                        source=rasterio.band(src, i),
                        destination=rasterio.band(dst, i),
                        src_transform=src.transform,
                        src_crs=src.crs,
                        dst_transform=ref_transform,
                        dst_crs=ref_crs,
                        resampling=Resampling.nearest
                    )
            print("✅ Aligned and compressed.")
    except Exception as e:
        print(f"❌ Error with {src_path}: {e}")



Processing: AutoRoute_03140201_240_cms_depth.tif
✅ Aligned and compressed.

Processing: HAND_03140201_240_cms_depth.tif
✅ Aligned and compressed.

Processing: AutoRoute_03140201_209_cms_depth.tif
✅ Aligned and compressed.

Processing: AutoRoute_03140201_130_cms_depth.tif
✅ Aligned and compressed.

Processing: HAND_03140201_209_cms_depth.tif
✅ Aligned and compressed.

Processing: HAND_03140201_272_cms_depth.tif
✅ Aligned and compressed.

Processing: HAND_03140201_165_cms_depth.tif
✅ Aligned and compressed.

Processing: AutoRoute_03140201_272_cms_depth.tif
✅ Aligned and compressed.

Processing: HAND_03140201_130_cms_depth.tif
✅ Aligned and compressed.

Processing: AutoRoute_03140201_165_cms_depth.tif
✅ Aligned and compressed.

Processing: HAND_03140201_79_cms_depth.tif
✅ Aligned and compressed.

Processing: AutoRoute_03140201_79_cms_depth.tif
✅ Aligned and compressed.


## Generate FIM source input file

 All input files inside the final_gis_files folder must start with one of the following valid model names:
HEC-RAS1D, HEC-RAS2D, HEC-RASCombo, SRH2D, FIER, AutoRoute, HAND, TRITON, Satellite, Surveyed, or Others.

In [ ]:
import os
import pandas as pd

# Folder path
input_folder = "/content/final_gis_files"

# Approved modelname mapping from short codes to full names
modelname_mapping = {
    "HEC-RAS1D": "HEC-RAS 1D",
    "HEC-RAS2D": "HEC-RAS 2D",
    "HEC-RASCombo": "HEC-RAS 1D/2D Combo",
    "SRH2D": "SRH-2D",
    "FIER": "FIER",
    "AutoRoute": "AutoRoute",
    "HAND": "HAND",
    "TRITON": "TRITON",
    "Satellite": "Satellite Observations",
    "Surveyed": "Surveyed Flood Extents",
    "Others": "Others"
}

# Track valid and invalid modelnames
valid_modelnames = set()
invalid_files = []

# Step 1: Scan all files and validate model names
for filename in os.listdir(input_folder):
    if filename.endswith((".tif", ".shp")):
        parts = filename.split("_")
        if len(parts) >= 5:
            modelname = parts[0]
            if modelname in modelname_mapping:
                valid_modelnames.add(modelname)
            else:
                invalid_files.append(filename)

# Step 2: Stop and warn if there are invalid files
if invalid_files:
    print("❌ Aborted: Some filenames contain unrecognized model names:")
    for f in invalid_files:
        print(f"   ⚠️ {f}")
    print("\n🔍 Please check the file names properly. The model name must match the approved list.")
else:
    # Step 3: Proceed to generate CSV if all model names are valid
    records = []
    for model in sorted(valid_modelnames):
        records.append({
            "FIMSourceName": modelname_mapping[model],
            "CoordinateReference": "",
            "EntityName": "",
            "EntityContactEmail": "",
            "VersionNumber": "",
            "YearCreated": "",
            "EventDate": "",
            "AdditionalModelNotes": "",
            "Software": modelname_mapping[model]  # ✅ Write full name (not short code)
        })

    df = pd.DataFrame(records, columns=[
        "FIMSourceName", "CoordinateReference", "EntityName", "EntityContactEmail",
        "VersionNumber", "YearCreated", "EventDate", "AdditionalModelNotes", "Software"
    ])
    output_path = "/content/FIM_input_data.csv"
    df.to_csv(output_path, index=False)
    print(f"✅ Input file generated: {output_path}")



✅ Input file generated: /content/FIM_input_data.csv


### Create your rating curve input files for each model

In [ ]:
import os
import re
import pandas as pd
from collections import defaultdict

# Directory containing all GIS files
input_folder = "/content/final_gis_files"

# Helper to parse file info
def parse_file_info(filename):
    match = re.match(r"([A-Za-z0-9\-]+)_(\d+)_(\d+)_([a-z]+)_(.+)\.(.+)", filename)
    if match:
        model, river_id, flow_str, unit, indicator, ext = match.groups()
        try:
            flow = int(flow_str)
            if unit.lower() == "cms":
                flow = round(flow * 35.3147, 0)
            return model, flow, indicator.lower(), ext.lower(), filename
        except:
            return None
    elif filename.endswith("boundary.shp"):
        parts = filename.split("_")
        model = parts[0]
        return model, None, "boundary", "shp", filename
    return None

# Organize files by model and flow
file_index = defaultdict(lambda: defaultdict(dict))  # file_index[model][flow][indicator] = filename

for fname in os.listdir(input_folder):
    parsed = parse_file_info(fname)
    if parsed:
        model, flow, indicator, ext, fname = parsed

        # Only accept .shp files for 'extent' and 'boundary'
        if indicator == "extent" and ext != "shp":
            continue
        if indicator == "boundary" and ext != "shp":
            continue

        if flow is not None:
            file_index[model][flow][indicator] = fname
        else:
            file_index[model]["boundary"]["boundary"] = fname

# Build and write rating curve CSVs per model
for model, flows in file_index.items():
    rows = []
    for flow, files in flows.items():
        if flow == "boundary":
            continue

        row = {
            "Flow": flow,
            "Depth": "",
            "ReturnPeriod": "",
            "VectorExtent": files.get("extent", ""),
            "DepthRaster": files.get("depth", ""),
            "WSERaster": files.get("wse", ""),
            "VelocityRaster": files.get("velocity", ""),
            "BoundaryVector": file_index[model].get("boundary", {}).get("boundary", "")
        }

        # Only add row if VectorExtent is .shp and DepthRaster exists
        if row["VectorExtent"].endswith(".shp") and row["DepthRaster"].endswith(".tif"):
            rows.append(row)

    if rows:
        df = pd.DataFrame(rows, columns=[
            "Flow", "Depth", "ReturnPeriod",
            "VectorExtent", "DepthRaster", "WSERaster", "VelocityRaster", "BoundaryVector"
        ])
        output_path = f"/content/{model}_ratingcurve.csv"
        df.to_csv(output_path, index=False)
        print(f"✅ Saved: {output_path}")
    else:
        print(f"⚠️ Skipped {model}: missing required files (extent.shp and depth.tif)")


✅ Saved: /content/HAND_ratingcurve.csv
✅ Saved: /content/AutoRoute_ratingcurve.csv


### Finally, download all the input files and folder

In [ ]:
import zipfile
import os
from google.colab import files

# Name of the zip file
zip_path = "/content/final_outputs.zip"

# Create zip
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:

    # Add all files in final_gis_files
    for root, _, files_in_dir in os.walk("/content/final_gis_files"):
        for file in files_in_dir:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, start="/content")
            zipf.write(full_path, arcname)

    # Add all .csv files in /content
    for file in os.listdir("/content"):
        if file.endswith(".csv") and file != os.path.basename(zip_path):
            zipf.write(os.path.join("/content", file), file)

# Download
files.download(zip_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>